# Stock Screener — Buy / Sell / Hold
Downloads S&P 500 data, calculates technical indicators, outputs a clean signal for every stock.

In [ ]:
# Install dependencies (run once)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'yfinance', 'pandas_ta', 'pandas', 'numpy', 'requests', 'lxml'], check=True)
print('Done')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta

# ── CONFIG ────────────────────────────────────────────────────────────────────
UNIVERSE   = 'sp500'   # 'sp500' | 'nasdaq100' | 'custom'
CUSTOM_TICKERS = []    # only used when UNIVERSE = 'custom'
PERIOD     = '6mo'     # data lookback for indicator calculations
THREADS    = True      # parallel download
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
def get_tickers(universe: str) -> list[str]:
    if universe == 'sp500':
        url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        df  = pd.read_html(url)[0]
        return df['Symbol'].str.replace('.', '-', regex=False).tolist()

    elif universe == 'nasdaq100':
        url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
        tables = pd.read_html(url)
        for t in tables:
            if 'Ticker' in t.columns or 'Symbol' in t.columns:
                col = 'Ticker' if 'Ticker' in t.columns else 'Symbol'
                return t[col].str.replace('.', '-', regex=False).tolist()

    elif universe == 'custom':
        return CUSTOM_TICKERS

    raise ValueError(f'Unknown universe: {universe}')

tickers = get_tickers(UNIVERSE)
print(f'Universe: {UNIVERSE} — {len(tickers)} tickers')

In [ ]:
print(f'Downloading {len(tickers)} tickers (batch) ...')
raw = yf.download(
    tickers,
    period=PERIOD,
    interval='1d',
    group_by='ticker',
    auto_adjust=True,
    threads=THREADS,
    progress=True
)
print('Download complete.')

In [ ]:
def compute_signals(ticker: str, raw_data) -> dict | None:
    try:
        if len(tickers) == 1:
            df = raw_data.copy()
        else:
            df = raw_data[ticker].dropna(how='all').copy()

        if len(df) < 60:
            return None

        df.columns = [c.lower() for c in df.columns]

        # ── Indicators ────────────────────────────────────────────────────────
        df.ta.rsi(length=14,   append=True)   # RSI_14
        df.ta.macd(append=True)               # MACD_12_26_9, MACDh_12_26_9, MACDs_12_26_9
        df.ta.bbands(length=20, append=True)  # BBL_20_2.0, BBM_20_2.0, BBU_20_2.0
        df.ta.sma(length=50,   append=True)   # SMA_50
        df.ta.sma(length=200,  append=True)   # SMA_200
        df.ta.atr(length=14,   append=True)   # ATRr_14

        last = df.iloc[-1]
        prev = df.iloc[-2]
        price = last['close']

        score = 0
        reasons = []

        # RSI
        rsi = last.get('RSI_14')
        if pd.notna(rsi):
            if rsi < 30:
                score += 1; reasons.append(f'RSI oversold ({rsi:.1f})')
            elif rsi > 70:
                score -= 1; reasons.append(f'RSI overbought ({rsi:.1f})')

        # MACD crossover
        macd_h     = last.get('MACDh_12_26_9')
        macd_h_prv = prev.get('MACDh_12_26_9')
        if pd.notna(macd_h) and pd.notna(macd_h_prv):
            if macd_h > 0 and macd_h_prv <= 0:
                score += 1; reasons.append('MACD bullish cross')
            elif macd_h < 0 and macd_h_prv >= 0:
                score -= 1; reasons.append('MACD bearish cross')
            elif macd_h > 0:
                score += 0.5
            else:
                score -= 0.5

        # Bollinger Bands
        bbl = last.get('BBL_20_2.0')
        bbu = last.get('BBU_20_2.0')
        if pd.notna(bbl) and pd.notna(bbu):
            if price < bbl:
                score += 1; reasons.append('Below lower BB')
            elif price > bbu:
                score -= 1; reasons.append('Above upper BB')

        # SMA 50/200 Golden / Death Cross
        sma50  = last.get('SMA_50')
        sma200 = last.get('SMA_200')
        if pd.notna(sma50) and pd.notna(sma200):
            if sma50 > sma200:
                score += 1; reasons.append('Golden cross (SMA50>200)')
            else:
                score -= 1; reasons.append('Death cross (SMA50<200)')

        # Price vs SMA50
        if pd.notna(sma50):
            if price > sma50:
                score += 0.5
            else:
                score -= 0.5

        # ── Decision ──────────────────────────────────────────────────────────
        if score >= 2:
            signal = 'BUY'
        elif score <= -2:
            signal = 'SELL'
        else:
            signal = 'HOLD'

        # 1-week price change
        chg1w = (price / df['close'].iloc[-6] - 1) * 100 if len(df) >= 6 else np.nan

        return {
            'Ticker': ticker,
            'Price':  round(price, 2),
            '1W %':   round(chg1w, 2),
            'RSI':    round(rsi, 1)  if pd.notna(rsi)  else np.nan,
            'Score':  round(score, 1),
            'Signal': signal,
            'Reasons': ' | '.join(reasons),
        }

    except Exception:
        return None


print('Computing signals ...')
results = [compute_signals(t, raw) for t in tickers]
results = [r for r in results if r is not None]
df_out  = pd.DataFrame(results).sort_values('Score', ascending=False).reset_index(drop=True)
print(f'Done — {len(df_out)} stocks scored')

In [ ]:
def color_signal(val):
    colors = {'BUY': 'background-color:#1a7a1a;color:white',
              'SELL': 'background-color:#9b1c1c;color:white',
              'HOLD': 'background-color:#5a5a00;color:white'}
    return colors.get(val, '')

def color_score(val):
    if val >= 2:   return 'color:#4ade80;font-weight:bold'
    if val <= -2:  return 'color:#f87171;font-weight:bold'
    return ''

summary = df_out.groupby('Signal')['Ticker'].count().rename('Count')
print('\n── Signal Summary ──')
print(summary.to_string())
print(f'\nGenerated: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')

(
    df_out.style
    .applymap(color_signal, subset=['Signal'])
    .applymap(color_score,  subset=['Score'])
    .format({'Price': '${:.2f}', '1W %': '{:+.2f}%', 'RSI': '{:.1f}', 'Score': '{:+.1f}'})
    .set_caption(f'Stock Screener — {UNIVERSE.upper()} — {datetime.now().strftime("%Y-%m-%d %H:%M")}')
)

In [ ]:
# ── Top BUYs and SELLs ───────────────────────────────────────────────────────
print('=== TOP 10 BUY SIGNALS ===')
display(df_out[df_out['Signal'] == 'BUY'].head(10)[['Ticker','Price','1W %','RSI','Score','Reasons']])

print('\n=== TOP 10 SELL SIGNALS ===')
display(df_out[df_out['Signal'] == 'SELL'].tail(10)[['Ticker','Price','1W %','RSI','Score','Reasons']])

In [ ]:
# ── Export to CSV ─────────────────────────────────────────────────────────────
out_file = f'signals_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
df_out.to_csv(out_file, index=False)
print(f'Saved → {out_file}')